# Local Earth Engine GeoTIFF export

This prototype authenticates Earth Engine locally, clips the GRACE mascon mean to the supplied High Plains Aquifer shapefile, and submits a non-blocking GeoTIFF export to Google Drive. It does not mount Drive, pull XEE data, or convert GeoTIFFs to NetCDF.

In [ ]:
%pip -q install earthengine-api geopandas shapely pyproj
from pathlib import Path
import json
import os
import time
import ee
import geopandas as gpd

PROJECT_ID = os.getenv('GEE_PROJECT', 'ee-ishansinhagzb')
DRIVE_FOLDER = os.getenv('EE_DRIVE_FOLDER', 'Ogallala_Phase0/01_gee_rasters/01a_grace_mascons/raw')
LOCAL_ROOT = Path(os.getenv('OGALLALA_PHASE0_ROOT', './Ogallala_Phase0'))
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)


In [ ]:
boundary_path = Path('high_plains_quifer/hp_bound2010.shp')
gdf = gpd.read_file(boundary_path).to_crs('EPSG:4326')
geometry = ee.Geometry(json.loads(gdf.to_json())['features'][0]['geometry'])

collection = (ee.ImageCollection('NASA/GRACE/MASS_GRIDS_V04/MASCON')
              .filterBounds(geometry)
              .filterDate('2002-04-01', '2024-09-30'))
image = collection.mean().clip(geometry)


In [ ]:
description = 'OG_01a_GRACE_TWSA_2002-04-01_2024-09-30_1000m'
task = ee.batch.Export.image.toDrive(
    image=image,
    description=description,
    folder=DRIVE_FOLDER,
    fileNamePrefix=description,
    region=geometry,
    scale=1000,
    crs='EPSG:5070',
    maxPixels=1e13,
    fileFormat='GeoTIFF',
)
task.start()
print({'task_id': task.id, 'status': task.status(), 'drive_folder': DRIVE_FOLDER})
